In [1]:
# Step 1: Install dependencies (run once)
!pip install pdfplumber pytesseract pdf2image pillow tqdm

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pdfplumber
from pdf2image import convert_from_path
import pytesseract
from tqdm import tqdm
import os
import re

In [3]:
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

In [4]:
def is_text_based(pdf_path, sample_pages=3):
    """Check first few pages to detect if PDF has text."""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages[:sample_pages]:
            text = page.extract_text()
            if text and len(text.strip()) > 50:
                return True
    return False

In [5]:
def extract_text_from_pdf_fast(pdf_path, skip_pages=2):
    print(f"📘 Reading file: {os.path.basename(pdf_path)}")

    # Quick check — is it text or image?
    text_based = is_text_based(pdf_path)
    mode = "Text" if text_based else "OCR"
    print(f"🔍 Detected mode: {mode}")

    extracted_text = ""

    if text_based:
        # ---- Text-based extraction ----
        with pdfplumber.open(pdf_path) as pdf:
            for page in tqdm(pdf.pages[skip_pages:], desc="Extracting text (pdfplumber)"):
                text = page.extract_text()
                if text:
                    extracted_text += text + "\n"
    else:
        # ---- OCR-based extraction ----
        print("⚙️ Converting pages to images for OCR...")
        pages = convert_from_path(pdf_path, dpi=200)
        for page in tqdm(pages[skip_pages:], desc="Running OCR"):
            text = pytesseract.image_to_string(page, lang="eng")
            extracted_text += text + "\n"

    print("✅ Extraction completed.")
    return extracted_text

def clean_text(text):
    """Remove page numbers, safe harbor notes, extra spaces."""
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'Page \d+ of \d+', '', text, flags=re.IGNORECASE)
    text = re.sub(r'Safe Harbor.*?Statement', '', text, flags=re.IGNORECASE)
    return text.strip()


In [6]:
pdf_path = r"THYROCARE_Q2FY26_Transcript.pdf"
text = extract_text_from_pdf_fast(pdf_path, skip_pages=2)
cleaned_text = clean_text(text)

output_file = os.path.splitext(pdf_path)[0] + "_cleaned.txt"
with open(output_file, "w", encoding="utf-8") as f:
    f.write(cleaned_text)

print(f"\n✅ Saved cleaned text to: {output_file}")
print(f"Total characters extracted: {len(cleaned_text):,}")

📘 Reading file: THYROCARE_Q2FY26_Transcript.pdf
🔍 Detected mode: Text


Extracting text (pdfplumber): 100%|██████████| 16/16 [00:01<00:00, 11.35it/s]

✅ Extraction completed.

✅ Saved cleaned text to: THYROCARE_Q2FY26_Transcript_cleaned.txt
Total characters extracted: 46,376
